In [ ]:
import geopandas as gpd
from rasterio.mask import mask
from pysheds.grid import Grid
from pysheds.sview import Raster
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import shape, Point, Polygon
from shapely.ops import unary_union, voronoi_diagram
from scipy.ndimage import gaussian_filter, distance_transform_edt
from rasterio.features import shapes
from tqdm import tqdm

In [ ]:
grid = Grid.from_raster("elevation/fairfax_dem_1m.tif")
dem = grid.read_raster("elevation/fairfax_dem_1m.tif")

In [ ]:
pits = grid.fill_pits(dem)
depressions = grid.fill_depressions(pits)
del pits
flats = grid.resolve_flats(depressions)
del depressions
flowdir = grid.flowdir(flats)
del flats

In [ ]:
import rasterio
with rasterio.open('flowdir.tif', "w",
    driver="GTiff",
    height=flowdir.shape[0],
    width=flowdir.shape[1],
    count=1,
    dtype=flowdir.dtype,
    crs=grid.crs,
    transform=grid.affine) as dst:
    dst.write(flowdir, 1)

In [ ]:
flowdir = grid.read_raster('flowdir.tif', nodata=0)

## Subcatchment Creation Using Recorded Inlets
Directly apply the recorded stormwater infrastructure data to identify points of urban inflow.

In [ ]:
nodes = gpd.read_file('nodes.geojson')
edges = gpd.read_file('edges.geojson')
drainage = nodes[nodes.node_type == 'infall']

## Identify Land Where Drainage Occurs

In [ ]:
Ainv = grid.affine.__invert__()
sink_patch = np.array([[2,4,8],
                       [1,-2,16],
                       [128,64,32]], dtype=flowdir.dtype)

subcatchments = np.zeros_like(flowdir, dtype=np.uint8)
mask = (dem == -9999)

area = np.sum(~mask)
subcatchment_area = 0

geometries = tqdm(drainage.geometry)

for k, source in enumerate(geometries):
    j, i = Ainv * (source.x, source.y)
    i = int(round(i))
    j = int(round(j))
    # modification without replacement was tested to reveal neglible difference in delineation
    flowdir[i-1:i+2, j-1:j+2] = sink_patch 

    sub = grid.catchment(x=j, y=i, fdir=flowdir, xytype='index')

    sub &= ~mask
    mask[i, j] = True 

    count = sub.sum()
    if count == 0:
        continue

    subcatchment_area += count

    subcatchments[sub] = 1
    mask |= sub


    geometries.set_description(
        f"Area covered - {subcatchment_area / area * 100:.2f}%"
    )
    if k % 1000 == 0:
        np.save("subcatchments.npy", subcatchments)


In [ ]:
polygons = []
arr = (subcatchments != 0).astype(np.uint8)
for geom, val in shapes(arr, mask=arr==1, transform=grid.affine):
    polygons.append(shape(geom))

In [ ]:
polygon_gdf = gpd.GeoDataFrame(
    {"geometry": polygons},
    crs=grid.crs
)

## Delineate drainage land for subcatchments by infalls

In [ ]:
merged_polygon = unary_union(polygon_gdf.geometry)

In [ ]:
infalls = nodes[nodes.node_type == "infall"].clip(polygon_gdf)

In [ ]:
voro = voronoi_diagram(infalls.geometry.union_all(), envelope=merged_polygon)

In [ ]:
result_polys = []
for idx, cell in tqdm(list(enumerate(voro.geoms, start=1))):
    clipped = cell.intersection(merged_polygon)
    if not clipped.is_empty:
        result_polys.append({
            "infall_id": idx,
            "geometry": clipped
        })

In [ ]:
catchment_gdf = gpd.GeoDataFrame(result_polys, crs=polygon_gdf.crs)

In [ ]:
subcatchment_gdf = catchment_gdf[catchment_gdf.area >= 100] # ARBITRARY: minimum coverage of 100 m^2
infalls = infalls.clip(subcatchment_gdf)
subcatchment_gdf = gpd.sjoin(subcatchment_gdf, infalls, predicate="covers").drop(columns='index_right').reset_index(drop=True).reset_index().rename(columns={'index': 'subcatchment_id'})

In [ ]:
impervious_surfaces = gpd.read_file("GIS/impervious_surfaces/2023_Countywide_Impervious.shp").to_crs(grid.crs)

In [ ]:
subcatchment_gdf['geometry'] = subcatchment_gdf.geometry.make_valid()
impervious_surfaces['geometry'] = impervious_surfaces.geometry.make_valid()
subcatchment_gdf['pct_impervious'] = subcatchment_gdf.geometry.apply(lambda geom: round(impervious_surfaces.clip(geom).area.sum() / geom.area * 100, 3))

In [ ]:
soils = gpd.read_file('GIS/SOIL.geojson').to_crs(grid.crs)

In [ ]:
soils['HYDRO_GROUP'] = soils['HYDRO_GROUP'].replace(['C/D', 'B/D'], 'D').replace('NA', 'C').fillna('C')
green_ampt_params = {
    "A":  {"Ksat": 40, "Suction": 60,  "IMD": 0.30},
    "B":  {"Ksat": 20, "Suction": 110, "IMD": 0.35},
    "C":  {"Ksat": 6,  "Suction": 150, "IMD": 0.40},
    "D":  {"Ksat": 1,  "Suction": 200, "IMD": 0.45}
}
soils[['Ksat', 'Suction', 'IMD']] = soils['HYDRO_GROUP'].apply(
    lambda g: pd.Series(green_ampt_params[g])
)

In [ ]:
joined = gpd.overlay(subcatchment_gdf, soils, how='intersection')
joined['area'] = joined.geometry.area
grouped = joined.groupby('subcatchment_id').apply(
    lambda df: pd.Series({
        "Ksat": (df["area"] * df["Ksat"]).sum() / df["area"].sum(),
        "Suction": (df["area"] * df["Suction"]).sum() / df["area"].sum(),
        "IMD": (df["area"] * df["IMD"]).sum() / df["area"].sum(),
    })
)
subcatchment_gdf = subcatchment_gdf.merge(grouped, on='subcatchment_id')

In [ ]:
# find the mean slope of every subcatchment to its drainage point
for i, sub in tqdm(list(subcatchment_gdf.iterrows())):
    subset = nodes.clip(sub.geometry)
    max_diff = subset.elevation.max() - subset.elevation.min()
    subcatchment_gdf.loc[i, 'slope'] = max(max_diff / (sub.geometry.area ** 0.5) * 100, 0.1) # minimum slope of 0.1%
subcatchment_gdf['slope'] = subcatchment_gdf['slope'].fillna(0.1)

In [ ]:
subcatchment_gdf = gpd.read_file('subcatchments.geojson')

In [ ]:
subcatchment_gdf = subcatchment_gdf.drop(['infall_id', 'subcatchment_id'], axis=1).reset_index(names='subcatchment_id')

In [ ]:
sump = gpd.read_file('GIS/SUMP.geojson')
sump = sump.to_crs(subcatchment_gdf.crs)
floodplain = gpd.read_file('GIS/FLOODPLAIN.geojson')
floodplain = floodplain.to_crs(subcatchment_gdf.crs)

In [ ]:
intersections = gpd.overlay(subcatchment_gdf, sump, how='intersection')
intersections['int_area'] = intersections.area
int_area_sum = intersections.groupby('subcatchment_id')['int_area'].sum()
subcatchment_gdf['pct_sump'] = (
    int_area_sum.reindex(subcatchment_gdf['subcatchment_id'], fill_value=0)
    / subcatchment_gdf.area * 100
).round(3)

In [ ]:
floodplain = floodplain.reset_index(names='floodplain_id')
near = gpd.sjoin(subcatchment_gdf, floodplain, how='left', predicate='intersects')
near_subs = near.dropna(subset='floodplain_id')['subcatchment_id']
subcatchment_gdf['near_floodplain'] = subcatchment_gdf.subcatchment_id.isin(near_subs)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
floodplain.plot(ax=ax)
subcatchment_gdf[subcatchment_gdf.near_floodplain].plot(ax=ax, color='red')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
subcatchment_gdf.plot(ax=ax, column='pct_impervious', legend=True)
# infalls.plot(ax=ax, color='red', markersize=0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Subcatchments')
plt.show()

In [ ]:
subcatchment_gdf.to_file("subcatchments1.geojson", driver="GeoJSON")